In [1]:
import os
import sys
sys.path.append('../')

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = (
    "3"  # 0 = all messages; 1 = filter out INFO; 2 = filter out INFO and WARNING; 3 = only errors
)

import time
import argparse
import numpy as np
import uuid
import tensorflow as tf

from tensorflow.keras.optimizers import Adam

tf.get_logger().setLevel("ERROR")

import json


# from models.data import DataGenerator
from engine.config import config as PARAMS
from engine.data import DataGenerator
from engine.model import build_unet_without_dose_layer, build_full_model
from DoseEngines import DoseEngine
from DoseEngines import ModelConfig
from engine.utils.plot_utils import Plotter
from engine.loss import (
    dose_loss,
    auxiliary_loss,
    TrainableLossWeightsNormalized,
    leafs_loss,
    mus_loss,
)
import engine.utils.tf_utils as tf_utils
import engine.utils.path_utils as path_utils
import engine.utils.comet_utils as comet_utils
import engine.utils.io_utils as io_utils
import engine.utils.data_utils as data_utils

from engine.utils.test_utils import debug_print, TestSetup


# tf.debugging.set_log_device_placement(True)  # Log device placement of operations
tf.debugging.set_log_device_placement(False)  # Log device placement of operations

/home/rd/anaconda3/envs/autoplan/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:404: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [4]:
config = ModelConfig(
    ct_array_shape=(128, 128, 320),
    downsampling_factor=tuple((2,2,4)),
    resolution=(0.3, 0.3, 0.3),
    field_size=(40, 40),
    number_of_leaf_pairs=60,
    tpr_20_10=0.72,
    number_of_cps=180,
)

In [5]:
notebook_dir = os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
data_path = os.path.join(parent_dir, "database/AUTORPT/")

gen_val = DataGenerator(
    data_path,
    "validating",
    False,
    1,
    constraints=PARAMS.constraints,
    is_debug=0,
    constraint_mode="fixed",
    is_normalize_weight=1,
)

gen_plot = DataGenerator(data_path, "plotting", True, 1)


Number of files: 83 in validating cohort
Number of files: 1 in plotting cohort


In [ ]:
T = TestSetup(parent_dir=parent_dir)
T.create_dummy(number_of_leaf_pairs=60, number_of_cps=3)

batch_size = 1

config = T.config
ct = T.ct
ct_np = np.expand_dims(ct, axis=0)
ct_np = np.repeat(ct_np, batch_size, axis=0)
y_mlc = T.leafs
y_mus = T.mus

print()
print()
print()

# TensorFlow: build layer and compute output.
tf_layer = AccumulateDose3DLayer(
    config,
    kernel_size=15,
    verbose=False,
)
ct_tf, y_mlc_tf, y_mus_tf = (
    tf.convert_to_tensor(ct_np),
    tf.convert_to_tensor(y_mlc),
    tf.convert_to_tensor(y_mus),
)

In [ ]:
# Make y_mlc_tf and y_mus_tf variables to optimize
leafs = tf.Variable(y_mlc_tf)
mus = tf.Variable(y_mus_tf)

# Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

In [ ]:
x, y_dose, masks, region_weights, constraints = gen_plot[0]

In [ ]:
@tf.function
def train_step():
    with tf.GradientTape() as tape:
        dose_pred = tf_layer(ct_tf, leafs, mus)
        loss_lb_gy, loss_hb_gy, loss_lb_target, loss_hb_target, l2_oars = dose_loss(
            x, dose_pred, constraints, masks, region_weights, None
        )
        aux_loss = 0
        mu_rate_loss, mu_complexity_loss = mus_loss(mus, config)
        leaf_opening_loss, leaf_rate_loss = leafs_loss(leafs, config)

        total_loss = (
            loss_lb_gy + loss_hb_gy + loss_lb_target + loss_hb_target +
            l2_oars + aux_loss + mu_rate_loss + mu_complexity_loss +
            leaf_opening_loss + leaf_rate_loss
        )

    grads = tape.gradient(total_loss, [leafs, mus])
    optimizer.apply_gradients(zip(grads, [leafs, mus]))
    return total_loss

# Run a few steps
for i in range(10):
    loss_val = train_step()
    print(f"Step {i}, Loss: {loss_val.numpy():.4f}")